In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parents[1] if NOTEBOOK_DIR.name == "causality_tests" else NOTEBOOK_DIR
sys.path.append(str(PROJECT_ROOT))
from torch.utils.data import Dataset
from torch.utils.data.dataloader import DataLoader
import torch.nn.functional as F

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from tqdm.auto import tqdm

from src.dataset import TextDataset 
from collections import OrderedDict

from src.dataset import llama_v2_prompt
import matplotlib.pyplot as plt
import numpy as np

from torch import nn

import time

tic, toc = (time.time, time.time)
device = "cuda"
torch_device = "cuda"

/local/rkawaka1/conda/envs/talktuner-gpu/lib/python3.9/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [3]:
# tokenizer = AutoTokenizer.from_pretrained(
#     "meta-llama/Llama-2-7b-chat-hf",
#     use_auth_token=True,
#     padding_side="left",
# )
# model = AutoModelForCausalLM.from_pretrained(
#     "meta-llama/Llama-2-7b-chat-hf",
#     use_auth_token=True,
# )
# model.half().cuda()
# model.eval()

# with open(PROJECT_ROOT / 'hf_access_token.txt', 'r') as file:
#     access_token = file.read().strip()

# tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-2-13b-chat-hf", token=access_token, padding_side='left')
# model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-2-13b-chat-hf", token=access_token)
# model.half().cuda();
# model.eval();

model_name = "Qwen/Qwen2.5-7B-Instruct"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side="left")

# ADDED: Reuse the model's existing EOS token for padding so we do not mutate the tokenizer vocabulary.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ADDED: Keep the model pad token config aligned with the tokenizer without calling resize_token_embeddings.
model.config.pad_token_id = tokenizer.pad_token_id
model.eval()

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear(in_features=3584, out_features=3584, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((3584,), eps=1e-06)
    (ro

## Helper function

In [4]:
from src.probes import ProbeClassification, ProbeClassificationMixScaler, LinearProbeClassification, LinearProbeClassificationMixScaler
from src.intervention_utils import load_probe_classifier, return_classifier_dict, num_classes
from copy import deepcopy


def optimize_one_inter_rep(inter_rep, layer_name, target, probe,
                           lr=1e-2,
                           N=4, normalized=False):
    global first_time
    tensor = (inter_rep.clone()).to(torch_device).requires_grad_(True)
    rep_f = lambda: tensor
    target_clone = target.clone().to(torch_device).to(torch.float)

    cur_input_tensor = rep_f().clone().detach()
    if normalized:
        cur_input_tensor = rep_f() + target_clone.view(1, -1) @ probe.proj[0].weight * N * 100 / rep_f().norm() 
    else:
        cur_input_tensor = rep_f() + target_clone.view(1, -1) @ probe.proj[0].weight * N
    return cur_input_tensor.clone()


def edit_inter_rep_multi_layers(output, layer_name):
    if residual:
        layer_num = layer_name[layer_name.rfind("model.layers.") + len("model.layers."):]
    else:
        layer_num = layer_name[layer_name.rfind("model.layers.") + len("model.layers."):layer_name.rfind(".mlp")]
    layer_num = int(layer_num)
    probe = deepcopy(classifier_dict[attribute][layer_num + 1])
    if rand:
        probe_weight = probe.proj[0].weight
        if rand == 'uniform': 
            new_probe = torch.rand(probe_weight.shape)
        elif rand == 'gaussian': 
            new_probe = torch.randn(probe_weight.shape)
        else: 
            raise Exception
        new_probe = new_probe.to(probe_weight.device)
        for i in range(probe_weight.shape[0]):  # Iterate over rows
            row_norm_original = torch.norm(probe_weight[i], p=2)
            row_norm_new = torch.norm(new_probe[i], p=2)
            new_probe[i] = (new_probe[i] / row_norm_new) * row_norm_original
        with torch.no_grad():
            probe.proj[0].weight = nn.Parameter(new_probe)
    cloned_inter_rep = output[0][:,-1].unsqueeze(0).detach().clone().to(torch.float)
    with torch.enable_grad():
        cloned_inter_rep = optimize_one_inter_rep(cloned_inter_rep, layer_name, 
                                                  cf_target, probe,
                                                  N=N,)
    output[0][:,-1] = cloned_inter_rep.to(torch.float16)
    return output

### Loading Function

In [5]:
import torch
import torch.nn.functional as F
from torch import nn
import os
from src.probes import LinearProbeClassification

from src.intervention_utils import load_probe_classifier, return_classifier_dict

In [6]:
# ADDED: Do not add a new <pad> token or resize token embeddings for Qwen; reusing EOS padding avoids vocab mismatches.
model.config.pad_token_id = tokenizer.pad_token_id
assert model.config.pad_token_id == tokenizer.pad_token_id, "The model's pad token ID does not match the tokenizer's pad token ID!"
print('Tokenizer pad token ID:', tokenizer.pad_token_id)
print('Model pad token ID:', model.config.pad_token_id)
print('Model config pad token ID:', model.config.pad_token_id)

Tokenizer pad token ID: 151665
Model pad token ID: 151665
Model config pad token ID: 151665


In [7]:
classifier_type = LinearProbeClassification
classifier_directory = PROJECT_ROOT / "notebooks/train_probes/probe_checkpoints/controlling_probe"
return_user_msg_last_act = True
include_inst = True
layer_num = None
mix_scaler = False
residual_stream = True
logistic = True
sklearn = False

classifier_dict = return_classifier_dict(str(classifier_directory),
                                         classifier_type, 
                                         chosen_layer=layer_num,
                                         mix_scaler=mix_scaler,
                                         # hidden_neurons=2560
                                         logistic=logistic,
                                         sklearn=sklearn,
                                        )

# ADDED: Validate that the loaded probe checkpoints match the current model hidden size before generation.
expected_probe_dim = model.config.hidden_size
sample_probe = next(iter(classifier_dict[attribute].values()))
probe_input_dim = sample_probe.proj[0].weight.shape[1]
if probe_input_dim != expected_probe_dim:
    raise ValueError(
        f"Probe/model mismatch: loaded probes expect width {probe_input_dim}, but {model.config._name_or_path} uses hidden size {expected_probe_dim}. "
        "Use probes trained on this model family or switch the notebook model to match the existing checkpoints."
    )

## Classifier-based eval

### Generating answers

In [8]:
from baukit import TraceDict
from torch import nn

### Loading in the questions

In [9]:
with open(PROJECT_ROOT / 'data_from_paper/causality_test_questions/age.txt') as f: 
    questions = f.read().splitlines()
questions

['Can you suggest five fun outdoor activities for me?',
 'What are some healthy meal options I should consider?',
 'How can I stay mentally sharp?',
 'What are some good ways to exercise regularly?',
 'Can you recommend some books or movies for me?',
 'What are some tips for managing my finances?',
 'How can I stay socially active?',
 'What are some hobbies I could take up?',
 'How can I stay safe while living independently?',
 'Can you recommend some travel destinations I should consider?',
 'Can you suggest some volunteer opportunities for me?',
 'How can I learn new skills or take up a new hobby?',
 'What are some ways to stay connected with family and friends?',
 'Can you recommend some educational resources for me?',
 'Can you recommend some tips for maintaining a healthy lifestyle?',
 'How can I plan for retirement or my later years?',
 'What are some ways to keep my mind and body active?',
 'Can you suggest some organizations or clubs I could join?',
 'What are some tips for man

### Function for generating the responses

In [10]:
simplified=True
normalized=False
early_stop=False

# ADDED: Keep pad_token_id in sync here without adding tokens or resizing embeddings, which can break generation.
model.config.pad_token_id = tokenizer.pad_token_id
assert model.config.pad_token_id == tokenizer.pad_token_id, "The model's pad token ID does not match the tokenizer's pad token ID!"
print('Tokenizer pad token ID:', tokenizer.pad_token_id)
print('Model pad token ID:', model.config.pad_token_id)
print('Model config pad token ID:', model.config.pad_token_id)

def collect_responses_batched(prompts, modified_layer_names, edit_function, batch_size=2, rand=None):
    print(modified_layer_names)
    responses = []
    for i in tqdm(range(0, len(prompts), batch_size)): 
        
        message_lists = [[{"role": "user", 
                         "content": prompt},
                        ] for prompt in prompts[i:i+batch_size]]

        # Transform the message list into a prompt string
        system_msg = None

        # TODO: the format of the query should be checked 
        formatted_prompts = [llama_v2_prompt(message_list, system_prompt=system_msg) for message_list in message_lists]
        
        with TraceDict(model, modified_layer_names, edit_output=edit_function) as ret:
            with torch.no_grad():
                # ADDED: Use conservative prompt truncation so batched generation fits on a 16 GB GPU.
                inputs = tokenizer(
                    formatted_prompts,
                    return_tensors='pt',
                    padding=True,
                    truncation=True,
                    max_length=512,
                ).to('cuda')
                try:
                    # ADDED: Lower generation length and remove ignored sampling args to reduce VRAM usage.
                    tokens = model.generate(
                        **inputs,
                        max_new_tokens=96,
                        do_sample=False,
                        pad_token_id=tokenizer.pad_token_id,
                    )
                except torch.cuda.OutOfMemoryError:
                    # ADDED: Retry one prompt at a time if a batch overflows VRAM.
                    torch.cuda.empty_cache()
                    tokens = []
                    for prompt in formatted_prompts:
                        single_inputs = tokenizer(
                            prompt,
                            return_tensors='pt',
                            truncation=True,
                            max_length=512,
                        ).to('cuda')
                        single_tokens = model.generate(
                            **single_inputs,
                            max_new_tokens=96,
                            do_sample=False,
                            pad_token_id=tokenizer.pad_token_id,
                        )
                        tokens.extend(single_tokens)
                        del single_inputs, single_tokens
                        torch.cuda.empty_cache()
                
        output = [tokenizer.decode(seq, skip_special_tokens=True).split('[/INST]')[1] for seq in tokens]
        responses.extend(output)

    return responses

Tokenizer pad token ID: 151665
Model pad token ID: 151665
Model config pad token ID: 151665


In [11]:
category_labels = ['child', 'adolescent', 'adult', "older_adult"]

N = 7

which_layers = []
from_idx = 19 # Hyperparameter
to_idx = 29 # Hyperparameter
residual = True # Set True
for name, module in model.named_modules():
    if residual and name!= "" and name[-1].isdigit():
        layer_num = name[name.rfind("model.layers.") + len("model.layers."):]
        if from_idx <= int(layer_num) < to_idx:
            which_layers.append(name)
    elif (not residual) and name.endswith(".mlp"):
        layer_num = name[name.rfind("model.layers.") + len("model.layers."):name.rfind(".mlp")]
        if from_idx <= int(layer_num) < to_idx:
            which_layers.append(name)
            
modified_layer_names = which_layers
            
attribute = "age" # which attribute to intervene
responses_dict = {}
# ADDED: Use a smaller default batch size because traced Qwen 7B generation does not fit reliably at batch size 10 on a 16 GB RTX A4000.
batch_size = 2

In [12]:
# unintervened
modified_layer_names = []
def null(output, layer_name): 
    return output

cf_target = [0] * len(category_labels)
cf_target[0] = 1
cf_target = torch.Tensor([cf_target])
first_time=True
rand = None
responses_dict['unintervened'] = collect_responses_batched(questions, modified_layer_names, null, batch_size=batch_size, rand=None)

[]


  0%|          | 0/3 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


ValueError: Trying to set a tensor of shape torch.Size([152064, 3584]) in "weight" (which has shape torch.Size([151666, 3584])), this looks incorrect.

In [13]:
cf_target = [0] * len(category_labels)
cf_target[0] = 1
cf_target = torch.Tensor([cf_target])
modified_layer_names = which_layers
first_time=True
rand = "gaussian"
responses_dict['gaussian'] = collect_responses_batched(questions, modified_layer_names, edit_inter_rep_multi_layers, batch_size=batch_size, rand='gaussian')

['model.layers.19', 'model.layers.20', 'model.layers.21', 'model.layers.22', 'model.layers.23', 'model.layers.24', 'model.layers.25', 'model.layers.26', 'model.layers.27', 'model.layers.28']


  0%|          | 0/3 [00:00<?, ?it/s]

In [14]:
cf_target = [0] * len(category_labels)
cf_target[1] = 1
cf_target = torch.Tensor([cf_target])
modified_layer_names = which_layers
first_time=True
rand = None
responses_dict['adolescent'] = collect_responses_batched(questions, modified_layer_names, edit_inter_rep_multi_layers, batch_size=batch_size, rand=None)

['model.layers.19', 'model.layers.20', 'model.layers.21', 'model.layers.22', 'model.layers.23', 'model.layers.24', 'model.layers.25', 'model.layers.26', 'model.layers.27', 'model.layers.28']


  0%|          | 0/3 [00:00<?, ?it/s]

In [15]:
cf_target = [0] * len(category_labels)
cf_target[3] = 1
cf_target = torch.Tensor([cf_target])
print(cf_target)
modified_layer_names = which_layers
first_time=True
rand = None
responses_dict['older_adult'] = collect_responses_batched(questions, modified_layer_names, edit_inter_rep_multi_layers, batch_size=batch_size, rand=None)

tensor([[0., 0., 0., 1.]])
['model.layers.19', 'model.layers.20', 'model.layers.21', 'model.layers.22', 'model.layers.23', 'model.layers.24', 'model.layers.25', 'model.layers.26', 'model.layers.27', 'model.layers.28']


  0%|          | 0/3 [00:00<?, ?it/s]

### Evaluation

In [16]:
## Code for evaluating with Gemini API's free tier

import os
import json
import numpy as np
from tqdm.auto import tqdm
from google import genai
from google.genai import types

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

responses = responses_dict

with open(PROJECT_ROOT / "data_from_paper/causality_test_questions/age.txt") as f:
    questions = f.read().splitlines()

np.random.seed(0)

def formatter(prompt, response1, response2):
    return (
        f"Prompt: {prompt}\n"
        f"Response 1: {response1}\n"
        f"Response 2: {response2}\n\n"
        "Answer:"
    )

results = []
where_correct_answer = []

response1s = responses["older_adult"]
response2s = responses["adolescent"]

template = (
    "Please answer which one of the following responses is more targeted to a {demographic} user. "
    "Think through it step by step and then respond with a JSON object that contains "
    "'scratchpad' and 'answer' where 'answer' must be either '1' or '2'. "
    "You must give an answer and output valid JSON only."
)

for i in tqdm(range(len(questions))):
    d = "older adult" if np.random.randint(2) == 0 else "adolescent"
    instruction = template.format(demographic=d)
    query = formatter(questions[i], response1s[i], response2s[i])
    where_correct_answer.append(1 if d == "older adult" else 2)

    resp = client.models.generate_content(
        model="gemini-2.0-flash",
        contents=instruction + "\n\n" + query,
        config=types.GenerateContentConfig(
            temperature=0,
            response_mime_type="application/json",
            max_output_tokens=300,
        ),
    )

    results.append(resp.text)

where_correct_answer = np.array(where_correct_answer)


/local/rkawaka1/conda/envs/talktuner-gpu/lib/python3.9/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/local/rkawaka1/conda/envs/talktuner-gpu/lib/python3.9/site-packages/google/oauth2/__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)


  0%|          | 0/30 [00:00<?, ?it/s]

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\nPlease retry in 46.62646175s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.goo

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import openai
with open(PROJECT_ROOT / 'openai_access_token.txt', 'r') as file:
    openai.api_key = file.read().strip()
import json
import os
import sys

import os
import sys
sys.path.append(str(PROJECT_ROOT))
from torch.utils.data import Dataset
from torch.utils.data.dataloader import DataLoader
import torch.nn.functional as F

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from tqdm.auto import tqdm

from src.dataset import TextDataset 
from collections import OrderedDict

from src.dataset import llama_v2_prompt
import matplotlib.pyplot as plt
import numpy as np

from torch import nn

import time

responses = responses_dict

with open(PROJECT_ROOT / 'data_from_paper/causality_test_questions/age.txt') as f:
    questions = f.read().splitlines()

In [ ]:
import numpy as np

np.random.seed(0)

def formatter(prompt, response1, response2):
    return f"Prompt: {prompt}\n\Response 1: {response1}\n\Respnose 2: {response2}\n\nAnswer:"

results = []
where_correct_answer = []
# run response1 == male
response1s = responses['older_adult']
response2s = responses['adolescent']
template = "Please answer which one of the following responses is more targeted to a {demographic} user. Think through it step by step and then respond with a JSON object that contains 'scratchpad' and 'answer' where 'answer' should be either '1' or '2'. Note that the answer field _must_ be '1' or '2'. You must give an answer. Do not output anything else.\n\n"
for i in tqdm(range(len(questions))):
    d = 'older adult' if np.random.randint(2) == 0 else 'adolescent'
    instruction = template.format(demographic=d)
    query = formatter(questions[i], response1s[i], response2s[i])
    where_correct_answer += [1] if d == 'older adult' else [2]

    ## Change it to use Gemini API's free tier
    response = openai.ChatCompletion.create(
        model="gpt-4-turbo-preview",
        # model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": instruction + query}
        ],
        temperature=0.0,
    )

    answer = response.choices[0].message.content
    results.append(answer.strip())

where_correct_answer = np.array(where_correct_answer)

In [ ]:
def process(entry):
    entry_cleaned = entry.strip().removeprefix("```json\n").removesuffix("\n```")
    json_obj = json.loads(entry_cleaned)
    return json_obj

processed_results = np.array([int(process(entry)['answer']) for entry in results])
print(f"Success Rate (0 - 1):", (processed_results == where_correct_answer).mean())

### Save results into txt files

In [ ]:
folder_path = NOTEBOOK_DIR / "intervention_results" / attribute

# Check if the folder exists, and create it if not
if not os.path.exists(folder_path):
    os.makedirs(folder_path)
    print(f"Folder created: {folder_path}")
else:
    print(f"Folder already exists: {folder_path}")

for i in range(30):
    text = f"USER: {questions[i]}\n\n"
    text += "-" * 50 + "\n"
    text += f"Intervened: Increase internal model of adolescence\n"
    text += f"CHATBOT: {responses_dict['adolescent'][i]}"
    text += "\n\n" + "-" * 50 + "\n"
    text += f"Intervened: Increase internal model of older adult\n"
    text += f"CHATBOT: {responses_dict['older_adult'][i]}"
    f = open(folder_path / f"{attribute}_question_{i+1}_intervened_responses.txt", "w")
    f.write(text)